In [1]:
# !pip install transformers datasets

## 1. 라이브러리 로드

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import Adam
from datasets import load_dataset
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 디바이스:', device)

사용 디바이스: cuda


## 2. 데이터 로드 (KLUE-NLI)

문장 간 관계분류(NLI)는 전제(premise)와 가설(hypothesis)이 주어졌을 때  
두 문장의 관계를 분류합니다.
- 0: entailment (함의)
- 1: neutral (중립)
- 2: contradiction (모순)

In [3]:
dataset = load_dataset('klue', 'nli')
train_data = dataset['train']
test_data  = dataset['validation']
print('훈련 데이터:', len(train_data), '/ 테스트 데이터:', len(test_data))
print('예시:', train_data[0])

README.md: 0.00B [00:00, ?B/s]

c:\AI\envs\pt\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\okss2\.cache\huggingface\hub\datasets--klue. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


nli/train-00000-of-00001.parquet:   0%|          | 0.00/1.83M [00:00<?, ?B/s]

nli/validation-00000-of-00001.parquet:   0%|          | 0.00/224k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24998 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

훈련 데이터: 24998 / 테스트 데이터: 3000
예시: {'guid': 'klue-nli-v1_train_00000', 'source': 'NSMC', 'premise': '힛걸 진심 최고다 그 어떤 히어로보다 멋지다', 'hypothesis': '힛걸 진심 최고로 멋지다.', 'label': 0}


## 3. 전처리

In [4]:
tokenizer = BertTokenizer.from_pretrained('klue/bert-base')
max_seq_len = 128

class NLIDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # premise + hypothesis 를 [SEP]으로 연결 (문장 쌍 입력)
        encoding = tokenizer(
            item['premise'],
            item['hypothesis'],
            max_length=max_seq_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'token_type_ids': encoding['token_type_ids'].squeeze(),
            'labels':         torch.tensor(item['label'])
        }

train_dataset = NLIDataset(train_data)
test_dataset  = NLIDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)
print('전처리 완료')

전처리 완료


## 4. 모델 & 학습

In [5]:
label_names = ['entailment', 'neutral', 'contradiction']

model = BertForSequenceClassification.from_pretrained('klue/bert-base', num_labels=3)
model.to(device)
optimizer = Adam(model.parameters(), lr=5e-5)

def evaluate():
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    print(f'Accuracy: {correct/total*100:.2f}%')

epochs = 3
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                        token_type_ids=token_type_ids, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f'Epoch {epoch+1} loss: {total_loss/len(train_loader):.4f}')
    evaluate()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

Epoch 1 loss: 0.5836
Accuracy: 77.90%


Epoch 2/3: 100%|██████████| 782/782 [29:49<00:00,  2.29s/it]


Epoch 2 loss: 0.3024
Accuracy: 80.10%


Epoch 3/3: 100%|██████████| 782/782 [29:51<00:00,  2.29s/it]


Epoch 3 loss: 0.1654
Accuracy: 79.97%


## 5. 예측

In [6]:
def nli_predict(premise, hypothesis):
    model.eval()
    encoding = tokenizer(
        premise, hypothesis,
        max_length=max_seq_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    token_type_ids = encoding['token_type_ids'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                        token_type_ids=token_type_ids)
    pred = torch.argmax(outputs.logits, dim=1).item()
    print(f'전제: {premise}')
    print(f'가설: {hypothesis}')
    print(f'관계: {label_names[pred]}\n')

nli_predict(
    '100명 중 3명이 사망했다.',
    '100명 중 97명이 생존했다.'
)
nli_predict(
    '오늘 날씨가 맑다.',
    '오늘 비가 많이 온다.'
)
nli_predict(
    '그 식당은 서울에 있다.',
    '그 식당은 한국에 있다.'
)

전제: 100명 중 3명이 사망했다.
가설: 100명 중 97명이 생존했다.
관계: contradiction

전제: 오늘 날씨가 맑다.
가설: 오늘 비가 많이 온다.
관계: contradiction

전제: 그 식당은 서울에 있다.
가설: 그 식당은 한국에 있다.
관계: contradiction

